# RAGAGENT - demo notebook

This notebook walks through the building blocks of the agent step by step.
It uses the same modules as `app.py`, just without the desktop UI.

**Authors:** Alejandro Magdiel, Jorge Valdes, Alvaro Gallego  
**Course:** UFV - Master in Data Analytics & AI - NLP, 2025


## 1. Setup

Make sure you have copied `.env.example` to `.env` and filled in your
`OPENAI_API_KEY` before running the cells below.


In [ ]:
# If you launch the notebook from the project root this is enough.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from ragagent import config
from ragagent.documents import load_corpus, list_supported_files
from ragagent.rag import RagIndex
from ragagent.tools import build_default_toolset
from ragagent.agent import build_agent

print('Docs folder:', config.DOCS_DIR)
print('Found files:')
for p in list_supported_files(config.DOCS_DIR):
    print(' -', p.name)


## 2. Document loading & cleaning

Each file is parsed with the right extractor (PyPDF2, python-docx or openpyxl)
and the extracted text is normalised (whitespace, hyphenation, junk characters).


In [ ]:
corpus = load_corpus(config.DOCS_DIR)
for name, text in corpus.items():
    print(f'{name}: {len(text)} chars')
    print('  preview:', text[:160], '...')


## 3. Build the FAISS index

`RagIndex` chunks every document, encodes the chunks with
`all-MiniLM-L6-v2`, and stores the vectors in a `faiss.IndexFlatL2`.


In [ ]:
index = RagIndex().build()
print(index.stats())


## 4. Try a pure-RAG retrieval


In [ ]:
for chunk in index.retrieve('What are the office hours?', k=3):
    print('---')
    print(chunk)


## 5. The full agent

The agent decides which tool to call for each question:
- `search_documents` for company-internal questions
- `search_web` for fresh public information
- `send_email` to send a plain-text email via Gmail SMTP


In [ ]:
agent = build_agent(index)

# Example 1: answered from the local documents
print(agent.run('What are ACME Robotics office hours?'))


In [ ]:
# Example 2: falls back to web search
print(agent.run('Who is the current Pope and when was he elected?'))


## 6. Sample questions

- *What is ACME-X1?*
- *What is the price of the ACME-X1 Charging Dock?*
- *How long does the battery last?*
- *Send an email to 'me@example.com' con el asunto 'Test' y con el mensaje 'Hello'*
